# dqi v2: own Reed-Solomon MBUC decoder + best submission so far (CPU, ~10 min)

Inputs (right panel, **+ Add Input**):
1. the competition;
2. the output of the notebook version whose `submission.zip` should be kept for the other tasks
   (the latest `dqi-tower` version: chemistry x1.719 + invsqrt x4.81 + dqi x1.419).

Settings: **Accelerator: None**. Run with **Save Version -> Save & Run All (Commit)**, then submit the new version's `submission.zip`.

The decoder solves the key equation with a constant schedule of 2 x 70 steps (divsteps on the reversed remainders, with
the quotients stashed in the registers freed by the division by x, then a replay of the transition matrices that pops them),
and uses tower-field GF(((2^3)^2)^2) arithmetic as AND networks uncomputed by measurement. Expected local result: PASS,
sha256 `f296d406aaab6fc3...`.

In [ ]:
# 1. Locate the inputs
import json, os, pathlib, subprocess, sys, zipfile

INPUT = pathlib.Path('/kaggle/input')
WORK = pathlib.Path('/kaggle/working')
comp = next(INPUT.glob('**/verifiers/dqi')).parents[1]
zips = [p for p in INPUT.rglob('submission.zip')]
print('competition:', comp)
print('zips       :', zips)
assert len(zips) == 1, 'attach exactly one notebook output with a submission.zip'
SOURCE = zips[0]

In [ ]:
# 2. Write the decoder sources
(WORK / 'tower.py').write_text('"""Tower-field GF(2^12) = GF(((2^3)^2)^2) arithmetic as bilinear AND networks for kickmix.\n\nEvery nonlinear step is a bilinear map out ^= B(X, Y) realised as a list of AND terms\n(xform, yform, outvec): the AND of a linear form of X with a linear form of Y is XORed\ninto the output bits in outvec. Each term costs exactly one Toffoli to compute and zero\nto uncompute (X-basis measurement plus CZ phase fix-ups), so the costs are:\n\n    GF(2^3) multiply      6 ANDs        GF(2^3) inverse        6 ANDs\n    GF(2^6) multiply     18 ANDs        GF(2^6) inverse       24 ANDs\n    GF(2^12) multiply    54 ANDs        GF(2^12) inverse      78 ANDs\n\nagainst 63 (multiply) and 315 (inverse with scaffold) for the kickmix built-ins.\n\nRegisters may store values in any GF(2)-basis: the term forms are expressed in tower\ncoordinates and converted to the register\'s own coordinates with the basis matrices.\n"""\nfrom __future__ import annotations\n\nimport functools\nimport random\n\nimport kickmix as km\n\nDEG = 12\n\n\n# ----------------------------------------------------------------------------------------\n# GF(2) linear algebra on bit-vectors stored as Python ints.\n# A matrix is a list of column vectors: M[j] is the image of basis vector e_j.\n\ndef mat_apply(cols, v):\n    out = 0\n    j = 0\n    while v:\n        if v & 1:\n            out ^= cols[j]\n        v >>= 1\n        j += 1\n    return out\n\n\ndef mat_inverse(cols, n):\n    """Inverse of an n x n GF(2) matrix given as columns."""\n    # Gaussian elimination on rows of [M | I].\n    rows = []\n    for i in range(n):\n        r = 0\n        for j in range(n):\n            if (cols[j] >> i) & 1:\n                r |= 1 << j\n        rows.append((r, 1 << i))\n    for c in range(n):\n        p = next(i for i in range(c, n) if (rows[i][0] >> c) & 1)\n        rows[c], rows[p] = rows[p], rows[c]\n        for i in range(n):\n            if i != c and (rows[i][0] >> c) & 1:\n                rows[i] = (rows[i][0] ^ rows[c][0], rows[i][1] ^ rows[c][1])\n    # rows[i][1] is row i of the inverse.\n    inv_cols = [0] * n\n    for i in range(n):\n        for j in range(n):\n            if (rows[i][1] >> j) & 1:\n                inv_cols[j] |= 1 << i\n    return inv_cols\n\n\ndef row_times_mat(row, cols):\n    """Row vector times matrix: (row . M)_j = parity(row & M[j])."""\n    out = 0\n    for j, c in enumerate(cols):\n        if bin(row & c).count(\'1\') & 1:\n            out |= 1 << j\n    return out\n\n\ndef identity(n):\n    return [1 << j for j in range(n)]\n\n\n# ----------------------------------------------------------------------------------------\n# Classical tower field.\n\ndef _mul_poly_mod(a, b, mod, deg):\n    r = 0\n    while b:\n        if b & 1:\n            r ^= a\n        b >>= 1\n        a <<= 1\n        if (a >> deg) & 1:\n            a ^= mod\n    return r\n\n\nGF8_MOD = 0b1011  # u^3 + u + 1\n\n\ndef mul8(a, b):\n    return _mul_poly_mod(a, b, GF8_MOD, 3)\n\n\ndef _trace8(x):\n    return (x ^ mul8(x, x) ^ mul8(mul8(x, x), mul8(x, x))) & 1  # trace lands in GF(2)\n\n\nLAMBDA = next(l for l in range(1, 8) if _trace8(l) == 1)  # v^2 + v + LAMBDA irreducible\n\n\ndef mul64(x, y):\n    x0, x1, y0, y1 = x & 7, x >> 3, y & 7, y >> 3\n    p0, p1 = mul8(x0, y0), mul8(x1, y1)\n    p2 = mul8(x0 ^ x1, y0 ^ y1)\n    return (p0 ^ mul8(LAMBDA, p1)) | ((p2 ^ p0) << 3)\n\n\ndef _pow64(x, e):\n    r = 1\n    while e:\n        if e & 1:\n            r = mul64(r, x)\n        x = mul64(x, x)\n        e >>= 1\n    return r\n\n\ndef _trace64(x):\n    t, s = 0, x\n    for _ in range(6):\n        t ^= s\n        s = mul64(s, s)\n    return t\n\n\nMU = next(m for m in range(1, 64) if _trace64(m) == 1)  # w^2 + w + MU irreducible\n\n\ndef mul4096(x, y):\n    x0, x1, y0, y1 = x & 63, x >> 6, y & 63, y >> 6\n    p0, p1 = mul64(x0, y0), mul64(x1, y1)\n    p2 = mul64(x0 ^ x1, y0 ^ y1)\n    return (p0 ^ mul64(MU, p1)) | ((p2 ^ p0) << 6)\n\n\ndef pow4096(x, e):\n    r = 1\n    while e:\n        if e & 1:\n            r = mul4096(r, x)\n        x = mul4096(x, x)\n        e >>= 1\n    return r\n\n\n@functools.lru_cache(maxsize=None)\ndef basis_matrices(degree: int = DEG):\n    """Returns (T, Tinv): T maps tower coordinates to kickmix GF(2^12) coordinates."""\n    gf = km.GF2Field(degree)\n    order = (1 << degree) - 1\n    # A primitive tower element g and its minimal polynomial over GF(2).\n    g = next(c for c in range(2, 1 << degree)\n             if all(pow4096(c, order // p) != 1 for p in (3, 5, 7, 13)))\n    powers = [1]\n    for _ in range(degree):\n        powers.append(mul4096(powers[-1], g))\n    # Find the GF(2) relation sum c_k g^k = 0 with c_12 = 1 by solving on bit vectors.\n    cols = powers[:degree]\n    inv = mat_inverse(cols, degree)\n    coeffs = mat_apply(inv, powers[degree])  # g^12 = sum coeffs_k g^k\n    # Find a root r of the same polynomial in the kickmix field.\n    def is_root(r):\n        acc, rk = 0, 1\n        for k in range(degree):\n            if (coeffs >> k) & 1:\n                acc ^= rk\n            rk = gf.mul(rk, r)\n        return acc == rk\n    r = next(c for c in range(2, 1 << degree) if is_root(c))\n    # Discrete log of tower basis vectors, then map g^k -> r^k.\n    log = {}\n    acc = 1\n    for k in range(order):\n        log[acc] = k\n        acc = mul4096(acc, g)\n    T = [gf.pow(r, log[1 << j]) for j in range(degree)]\n    Tinv = mat_inverse(T, degree)\n    # Sanity: the map is a ring homomorphism.\n    rng = random.Random(1)\n    for _ in range(300):\n        a, b = rng.randrange(1 << degree), rng.randrange(1 << degree)\n        assert mat_apply(T, mul4096(a, b)) == gf.mul(mat_apply(T, a), mat_apply(T, b))\n    return T, Tinv\n\n\n# ----------------------------------------------------------------------------------------\n# Bilinear term lists in tower coordinates: (xform, yform, outvec).\n\ndef _const_mul_matrix(mul, c, n):\n    return [mul(c, 1 << j) for j in range(n)]\n\n\n@functools.lru_cache(maxsize=None)\ndef terms_mul8():\n    # Karatsuba-style 6-AND product of 3-bit polynomials, reduced modulo u^3 + u + 1.\n    # Raw product coefficients d0..d4 from p_i = a_i b_i and p_ij = (a_i + a_j)(b_i + b_j).\n    prods = {\n        \'p0\': (0b001, 0b001), \'p1\': (0b010, 0b010), \'p2\': (0b100, 0b100),\n        \'p01\': (0b011, 0b011), \'p02\': (0b101, 0b101), \'p12\': (0b110, 0b110),\n    }\n    # d0 = p0; d1 = p01+p0+p1; d2 = p02+p0+p1+p2; d3 = p12+p1+p2; d4 = p2.\n    d = [0] * 5\n    contrib = {\n        \'p0\': [0, 1, 2], \'p1\': [1, 2, 3], \'p2\': [2, 3, 4],\n        \'p01\': [1], \'p02\': [2], \'p12\': [3],\n    }\n    reduce = {0: 0b001, 1: 0b010, 2: 0b100, 3: 0b011, 4: 0b110}  # u^3 = u+1, u^4 = u^2+u\n    out = []\n    for name, (xf, yf) in prods.items():\n        vec = 0\n        for k in contrib[name]:\n            vec ^= reduce[k]\n        out.append((xf, yf, vec))\n    _check_terms(out, 3, 3, lambda a, b: mul8(a, b))\n    return tuple(out)\n\n\ndef _lift(terms, xsh, ysh, outmap):\n    """Shifts forms and pushes outvec through the linear map `outmap` (callable)."""\n    return [(xf << xsh, yf << ysh, outmap(ov)) for xf, yf, ov in terms]\n\n\ndef _ext_mul_terms(base, n, const_mul):\n    """Karatsuba over a quadratic extension z^2 = z + c of a base field of n bits."""\n    lo = lambda ov: ov\n    hi = lambda ov: ov << n\n    out = []\n    # p0 = x0 y0 contributes to both the constant and z coefficients.\n    out += [(xf, yf, ov | (ov << n)) for xf, yf, ov in base]\n    # p1 = x1 y1 contributes c * p1 to the constant coefficient.\n    cm = [const_mul(1 << j) for j in range(n)]\n    out += [(xf << n, yf << n, mat_apply(cm, ov)) for xf, yf, ov in base]\n    # p2 = (x0 + x1)(y0 + y1) contributes to the z coefficient.\n    out += [(xf | (xf << n), yf | (yf << n), ov << n) for xf, yf, ov in base]\n    return out\n\n\n@functools.lru_cache(maxsize=None)\ndef terms_mul64():\n    t = _ext_mul_terms(terms_mul8(), 3, lambda v: mul8(LAMBDA, v))\n    _check_terms(t, 6, 6, mul64)\n    return tuple(t)\n\n\n@functools.lru_cache(maxsize=None)\ndef terms_mul4096():\n    t = _ext_mul_terms(terms_mul64(), 6, lambda v: mul64(MU, v))\n    _check_terms(t, 12, 12, mul4096)\n    return tuple(t)\n\n\ndef _eval_terms(terms, x, y):\n    out = 0\n    for xf, yf, ov in terms:\n        if (bin(xf & x).count(\'1\') & 1) and (bin(yf & y).count(\'1\') & 1):\n            out ^= ov\n    return out\n\n\ndef _check_terms(terms, nx, ny, fn, samples=400):\n    rng = random.Random(7)\n    pairs = [(rng.randrange(1 << nx), rng.randrange(1 << ny)) for _ in range(samples)]\n    if nx + ny <= 12:\n        pairs = [(a, b) for a in range(1 << nx) for b in range(1 << ny)]\n    for a, b in pairs:\n        assert _eval_terms(terms, a, b) == fn(a, b), (a, b)\n\n\ndef precompose(terms, xmap=None, ymap=None, outmap=None):\n    """Rewrites tower-coordinate terms for registers stored in other bases.\n\n    xmap/ymap: row-transform matrices (columns) R such that tower = R . register,\n    i.e. a tower form l becomes l . R on register bits. outmap: matrix (columns)\n    taking tower output vectors to register output vectors.\n    """\n    out = []\n    for xf, yf, ov in terms:\n        if xmap is not None:\n            xf = row_times_mat(xf, xmap)\n        if ymap is not None:\n            yf = row_times_mat(yf, ymap)\n        if outmap is not None:\n            ov = mat_apply(outmap, ov)\n        out.append((xf, yf, ov))\n    return out\n\n\n# ----------------------------------------------------------------------------------------\n# Quantum emitters.\n\ndef _prep(builder, reg, form):\n    """CNOT-folds the linear form onto its lowest qubit; returns (pivot, undo-list)."""\n    bits = [j for j in range(len(reg)) if (form >> j) & 1]\n    p = bits[0]\n    for j in bits[1:]:\n        builder.cx(reg[j], reg[p])\n    return p, bits[1:]\n\n\ndef _unprep(builder, reg, p, rest):\n    for j in reversed(rest):\n        builder.cx(reg[j], reg[p])\n\n\ndef emit_bilinear(builder, X, Y, out, terms, scratch):\n    """out ^= B(X, Y). X, Y, out must be disjoint qubit arrays. One Toffoli per term."""\n    for xf, yf, ov in terms:\n        if not ov or not xf or not yf:\n            continue\n        px, rx = _prep(builder, X, xf)\n        py, ry = _prep(builder, Y, yf)\n        targets = [out[k] for k in range(len(out)) if (ov >> k) & 1]\n        if len(targets) == 1:\n            builder.ccx(X[px], Y[py], targets[0])\n        else:\n            builder.init_and(X[px], Y[py], scratch)\n            builder.cx(scratch, km.array(targets))\n            builder.del_and(X[px], Y[py], scratch)\n        _unprep(builder, Y, py, ry)\n        _unprep(builder, X, px, rx)\n\n\ndef emit_bilinear_mbuc_clear(builder, X, Y, out, terms):\n    """Clears `out` (holding exactly B(X, Y)) by X-measurement and CZ phase fix-ups."""\n    bits = builder.alloc_bits(len(out))\n    builder.hmr(out, bits)\n    for k in range(len(out)):\n        sel = [(xf, yf) for xf, yf, ov in terms if (ov >> k) & 1 and xf and yf]\n        if not sel:\n            continue\n        builder.push_condition(bits[k])\n        for xf, yf in sel:\n            px, rx = _prep(builder, X, xf)\n            py, ry = _prep(builder, Y, yf)\n            builder.cz(X[px], Y[py])\n            _unprep(builder, Y, py, ry)\n            _unprep(builder, X, px, rx)\n        builder.pop_condition()\n    builder.free(bits)\n\n\ndef emit_linear(builder, src, dst, cols):\n    """dst ^= M . src, M given by columns (cols[j] = image of src bit j)."""\n    for j, c in enumerate(cols):\n        for k in range(len(dst)):\n            if (c >> k) & 1:\n                builder.cx(src[j], dst[k])\n\n\n# ----------------------------------------------------------------------------------------\n# Tower inverse.\n\ndef _sq_matrix(mul, n):\n    return [mul(1 << j, 1 << j) for j in range(n)]\n\n\n@functools.lru_cache(maxsize=None)\ndef _inverse_tables():\n    T, Tinv = basis_matrices()\n    m64 = terms_mul64()\n    m8 = terms_mul8()\n    # N = a*b + a^2 + MU*b^2 (6 bits), x = a + b w.\n    N_bil = [(xf, yf, ov) for xf, yf, ov in m64]  # X = a (6), Y = b (6)\n    sq64 = _sq_matrix(mul64, 6)\n    N_lin_a = sq64\n    N_lin_b = [mul64(MU, c) for c in sq64]\n    # n = c*d + c^2 + LAMBDA*d^2 (3 bits), N = c + d v.\n    sq8 = _sq_matrix(mul8, 3)\n    n_lin_c = sq8\n    n_lin_d = [mul8(LAMBDA, c) for c in sq8]\n    # ninv = n^4 * n^2, written as bilinear(n, ncopy) with forms precomposed by the squarings.\n    sq8_2 = [mat_apply(sq8, c) for c in sq8]  # x -> x^4\n    # terms over (z1 = x^4, z2 = x^2): l . z1 = l . (S4 x) -> row l times S4.\n    ninv_terms = [(row_times_mat(xf, sq8_2), row_times_mat(yf, sq8), ov) for xf, yf, ov in m8]\n    # Ninv = ((c + d) * ninv) + (d * ninv) v. X = N (6 bits), Y = ninv (3 bits).\n    Ninv_terms = []\n    for xf, yf, ov in m8:\n        Ninv_terms.append((xf | (xf << 3), yf, ov))        # (c + d) * ninv -> low\n        Ninv_terms.append((xf << 3, yf, ov << 3))          # d * ninv -> high\n    # y = ((a + b) * Ninv) + (b * Ninv) w. X = x (12 bits), Y = Ninv (6 bits).\n    y_terms = []\n    for xf, yf, ov in m64:\n        y_terms.append((xf | (xf << 6), yf, ov))\n        y_terms.append((xf << 6, yf, ov << 6))\n    return dict(N_bil=N_bil, N_lin_a=N_lin_a, N_lin_b=N_lin_b, n_lin_c=n_lin_c,\n                n_lin_d=n_lin_d, ninv=ninv_terms, Ninv=Ninv_terms, y=y_terms)\n\n\nclass TowerInverse:\n    """y = gate ? x^{-1} : 0 for x in a kickmix-basis register; 78 (+3 if gated) Toffolis."""\n\n    def __init__(self, builder, x, *, gate=None, out_basis=\'kickmix\'):\n        self.b = builder\n        self.x = x\n        self.gate = gate\n        T, Tinv = basis_matrices()\n        tb = _inverse_tables()\n        self.tb = tb\n        self.out_map = T if out_basis == \'kickmix\' else None\n        self.scratch = builder.alloc_qubits(1)[0]\n        # Express x in tower coordinates in place (free), restored by `release`.\n        self._to_tower = _cnot_program(Tinv)\n        _run_program(builder, x, self._to_tower)\n\n    def compute(self):\n        b, x, tb = self.b, self.x, self.tb\n        a_q, b_q = x[0:6], x[6:12]\n        self.N = b.alloc_qubits(6)\n        emit_bilinear(b, a_q, b_q, self.N, tb[\'N_bil\'], self.scratch)\n        emit_linear(b, a_q, self.N, tb[\'N_lin_a\'])\n        emit_linear(b, b_q, self.N, tb[\'N_lin_b\'])\n        c_q, d_q = self.N[0:3], self.N[3:6]\n        self.n = b.alloc_qubits(3)\n        emit_bilinear(b, c_q, d_q, self.n, terms_mul8(), self.scratch)\n        emit_linear(b, c_q, self.n, tb[\'n_lin_c\'])\n        emit_linear(b, d_q, self.n, tb[\'n_lin_d\'])\n        if self.gate is not None:\n            self.ng = b.alloc_qubits(3)\n            b.init_and(self.gate, self.n, self.ng)\n        else:\n            self.ng = self.n\n        self.ncopy = b.alloc_qubits(3)\n        b.cx(self.ng, self.ncopy)\n        self.ninv = b.alloc_qubits(3)\n        emit_bilinear(b, self.ng, self.ncopy, self.ninv, tb[\'ninv\'], self.scratch)\n        self.Ninv = b.alloc_qubits(6)\n        emit_bilinear(b, self.N, self.ninv, self.Ninv, tb[\'Ninv\'], self.scratch)\n        self.y = b.alloc_qubits(12)\n        y_terms = tb[\'y\'] if self.out_map is None else precompose(tb[\'y\'], outmap=self.out_map)\n        self.y_terms = y_terms\n        emit_bilinear(b, x, self.Ninv, self.y, y_terms, self.scratch)\n        return self.y\n\n    def uncompute(self):\n        b, x, tb = self.b, self.x, self.tb\n        emit_bilinear_mbuc_clear(b, x, self.Ninv, self.y, self.y_terms)\n        b.free(self.y)\n        emit_bilinear_mbuc_clear(b, self.N, self.ninv, self.Ninv, tb[\'Ninv\'])\n        b.free(self.Ninv)\n        emit_bilinear_mbuc_clear(b, self.ng, self.ncopy, self.ninv, tb[\'ninv\'])\n        b.free(self.ninv)\n        b.cx(self.ng, self.ncopy)\n        b.free(self.ncopy)\n        if self.gate is not None:\n            b.del_and(self.gate, self.n, self.ng)\n            b.free(self.ng)\n        c_q, d_q = self.N[0:3], self.N[3:6]\n        emit_linear(b, d_q, self.n, tb[\'n_lin_d\'])\n        emit_linear(b, c_q, self.n, tb[\'n_lin_c\'])\n        emit_bilinear_mbuc_clear(b, c_q, d_q, self.n, terms_mul8())\n        b.free(self.n)\n        a_q, b_q = x[0:6], x[6:12]\n        emit_linear(b, b_q, self.N, tb[\'N_lin_b\'])\n        emit_linear(b, a_q, self.N, tb[\'N_lin_a\'])\n        emit_bilinear_mbuc_clear(b, a_q, b_q, self.N, tb[\'N_bil\'])\n        b.free(self.N)\n\n    def release(self):\n        _run_program(self.b, self.x, _invert_program(self._to_tower))\n        self.b.free(self.scratch)\n\n\n# ----------------------------------------------------------------------------------------\n# In-place invertible linear maps as CNOT programs.\n\ndef _cnot_program(cols):\n    """CNOT list (control, target) realising v -> M v in place (M invertible, columns)."""\n    n = len(cols)\n    # Reduce M to I with row ops; row op "row_t ^= row_c" == CNOT(c -> t) acting on v.\n    rows = []\n    for i in range(n):\n        r = 0\n        for j in range(n):\n            if (cols[j] >> i) & 1:\n                r |= 1 << j\n        rows.append(r)\n    ops = []\n    for c in range(n):\n        if not (rows[c] >> c) & 1:\n            p = next(i for i in range(c + 1, n) if (rows[i] >> c) & 1)\n            rows[c] ^= rows[p]\n            ops.append((p, c))\n        for i in range(n):\n            if i != c and (rows[i] >> c) & 1:\n                rows[i] ^= rows[c]\n                ops.append((c, i))\n    # E_k ... E_1 M = I  =>  M = E_1 ... E_k (each self-inverse); apply E_k first.\n    return list(reversed(ops))\n\n\ndef _invert_program(ops):\n    return list(reversed(ops))\n\n\ndef _run_program(builder, reg, ops):\n    for c, t in ops:\n        builder.cx(reg[c], reg[t])\n\n\ndef emit_mul_tower(builder, X, Y, out, scratch, *, x_basis=\'kickmix\', y_basis=\'kickmix\',\n                   out_basis=\'kickmix\'):\n    """out ^= X * Y with 54 Toffolis; each register may be in kickmix or tower basis."""\n    T, Tinv = basis_matrices()\n    terms = precompose(\n        terms_mul4096(),\n        xmap=Tinv if x_basis == \'kickmix\' else None,\n        ymap=Tinv if y_basis == \'kickmix\' else None,\n        outmap=T if out_basis == \'kickmix\' else None,\n    )\n    emit_bilinear(builder, X, Y, out, terms, scratch)\n    return terms\n')
(WORK / 'dqi_decoder.py').write_text('"""DQI decoder v2: constant-schedule reversible key-equation solver + tower-field Forney.\n\nReplaces the reference\'s synchronized extended Euclidean algorithm (222 iterations, each paying\nfor every sub-step under quantum control) with two phases of exactly n = 70 steps whose shape is\nknown classically:\n\nPhase 1 (Bernstein-Yang style divsteps on the reversed remainders f, g):\n    swap_t = (delta > 0) and g(0) != 0;  if swap_t: (f, g) <- (g, f), delta <- -delta\n    c_t = g(0) / f(0);  g <- (g + c_t f) / x;  delta <- delta + 1\n  The division by x is a relabeling of g\'s registers, and the register freed by it stores c_t\n  (the stash), so each step is reversible with zero bookkeeping cost.\n\nPhase 2 (replay of the transition matrices from the right, popping the stash):\n    v <- v T_t P_t   with T_t = [[x, 0], [c_t, 1]] and P_t the swap, starting from v = (0, 1).\n  The new constant coefficient of v_f equals c_t v_g(0), so c_t is uncomputed from it.\n  At the end v = (omega_rev, sigma_rev).\n\nForney: sigma_rev(X) = 0 marks an error at X = alpha^j, with value X^-70 omega_rev(X) / (X sigma_rev\'(X)).\n\nRegister sizes per step come from classical simulation of the algorithm (bounds_k20.json):\nshots with error weight below 20 (probability ~1e-5) may fail.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport kickmix as km\n\nimport tower\n\nN_SYN = 70\nDEG = 12\nDELTA_BITS = 7\nBOUNDS = json.loads((Path(__file__).with_name(\'bounds_k20.json\')).read_text())\n\nT_MAT, TINV_MAT = tower.basis_matrices()\nTO_TOWER = tower._cnot_program(TINV_MAT)\nFROM_TOWER = tower._invert_program(TO_TOWER)\nMUL_TK_K = tower.precompose(tower.terms_mul4096(), ymap=TINV_MAT, outmap=T_MAT)  # tower x kickmix\nMUL_KT_K = tower.precompose(tower.terms_mul4096(), xmap=TINV_MAT, outmap=T_MAT)  # kickmix x tower\n\n\ndef _to_tower(b, reg):\n    tower._run_program(b, reg, TO_TOWER)\n\n\ndef _from_tower(b, reg):\n    tower._run_program(b, reg, FROM_TOWER)\n\n\nclass ZeroTest:\n    """flag = (reg == 0) by an AND tree, uncomputed by measurement."""\n\n    def __init__(self, builder, reg):\n        self.b, self.reg = builder, reg\n\n    def compute(self):\n        b = self.b\n        b.x(self.reg)\n        layer, self.steps = list(self.reg), []\n        while len(layer) > 1:\n            nxt = []\n            for i in range(0, len(layer) - 1, 2):\n                t = b.alloc_qubits(1)[0]\n                b.init_and(layer[i], layer[i + 1], t)\n                self.steps.append((layer[i], layer[i + 1], t))\n                nxt.append(t)\n            if len(layer) % 2:\n                nxt.append(layer[-1])\n            layer = nxt\n        b.x(self.reg)\n        return layer[0]\n\n    def uncompute(self):\n        b = self.b\n        b.x(self.reg)\n        for c1, c2, t in reversed(self.steps):\n            b.del_and(c1, c2, t)\n            b.free(t)\n        b.x(self.reg)\n\n\nclass Pool:\n    """LIFO pool of zeroed 12-qubit registers; keeps register identities stable under reversal."""\n\n    def __init__(self, builder):\n        self.b, self.free = builder, []\n\n    def acquire(self):\n        return self.free.pop() if self.free else self.b.alloc_qubits(DEG)\n\n    def release(self, reg):\n        self.free.append(reg)\n\n    def resize(self, lst, n):\n        while len(lst) < n:\n            lst.append(self.acquire())\n        while len(lst) > n:\n            self.release(lst.pop())\n\n\nclass TwoPhaseDecoder:\n\n    def __init__(self, builder, syndromes, error_mbuc):\n        self.b = builder\n        self.S = syndromes\n        self.E = error_mbuc\n        self.pool = Pool(builder)\n        self.scratch = None\n\n    # ------------------------------------------------------------------ helpers\n    def _mul(self, x, y, out, terms):\n        tower.emit_bilinear(self.b, x, y, out, terms, self.scratch)\n\n    def _swap_flags(self):\n        """Returns (pos, gz, cleanup) with pos = delta > 0 and gz = (g0 == 0)."""\n        b = self.b\n        ztd = ZeroTest(b, self.delta)\n        dz = ztd.compute()\n        ztg = ZeroTest(b, self.G[0])\n        gz = ztg.compute()\n        sign = self.delta[DELTA_BITS - 1]\n        pos = b.alloc_qubits(1)[0]\n        b.x(km.array([sign, dz]))\n        b.init_and(sign, dz, pos)\n        b.x(km.array([sign, dz]))\n\n        def cleanup():\n            b.x(km.array([sign, dz]))\n            b.del_and(sign, dz, pos)\n            b.x(km.array([sign, dz]))\n            b.free(pos)\n            ztg.uncompute()\n            ztd.uncompute()\n        return pos, gz, cleanup\n\n    # ------------------------------------------------------------------ phase 1\n    def _p1(self, t):\n        b, B = self.b, BOUNDS\n        self.p1_sizes.append((len(self.F), len(self.G)))\n        self.pool.resize(self.F, B[\'AF\'][t])\n        self.pool.resize(self.G, B[\'AG\'][t])\n        sb = b.alloc_qubits(1)[0]\n        pos, gz, cleanup = self._swap_flags()\n        b.x(gz)\n        b.init_and(pos, gz, sb)\n        b.x(gz)\n        cleanup()\n        for i in range(B[\'W\'][t]):\n            b.cswap(sb, self.F[i], self.G[i])\n        b.cx(sb, self.delta)\n        b.iadd(1, target=self.delta)\n        b.iadd(1, target=self.delta, control=sb)\n        c = b.alloc_qubits(DEG)\n        inv = tower.TowerInverse(b, self.F[0], out_basis=\'tower\')\n        y = inv.compute()\n        self._mul(self.G[0], y, c, MUL_KT_K)\n        inv.uncompute()\n        inv.release()\n        _to_tower(b, c)\n        for i in range(B[\'R\'][t]):\n            self._mul(c, self.F[i], self.G[i], MUL_TK_K)\n        _from_tower(b, c)\n        slot = self.G.pop(0)\n        b.cx(c, slot)\n        b.cx(slot, c)\n        b.free(c)\n        self.stash[t] = slot\n        self.swapbits[t] = sb\n\n    def _p1_rev(self, t):\n        b, B = self.b, BOUNDS\n        slot, sb = self.stash.pop(t), self.swapbits[t]\n        c = b.alloc_qubits(DEG)\n        b.cx(slot, c)\n        b.cx(c, slot)\n        self.G.insert(0, slot)\n        _to_tower(b, c)\n        for i in reversed(range(B[\'R\'][t])):\n            self._mul(c, self.F[i], self.G[i], MUL_TK_K)\n        _from_tower(b, c)\n        inv = tower.TowerInverse(b, self.F[0], out_basis=\'tower\')\n        y = inv.compute()\n        self._mul(self.G[0], y, c, MUL_KT_K)\n        inv.uncompute()\n        inv.release()\n        b.free(c)\n        b.isub(1, target=self.delta, control=sb)\n        b.isub(1, target=self.delta)\n        b.cx(sb, self.delta)\n        for i in reversed(range(B[\'W\'][t])):\n            b.cswap(sb, self.F[i], self.G[i])\n        pos, gz, cleanup = self._swap_flags()\n        b.x(gz)\n        b.del_and(pos, gz, sb)\n        b.x(gz)\n        cleanup()\n        b.free(sb)\n        nf, ng = self.p1_sizes.pop()\n        self.pool.resize(self.G, ng)\n        self.pool.resize(self.F, nf)\n\n    # ------------------------------------------------------------------ phase 2\n    def _p2(self, j):\n        b, B = self.b, BOUNDS\n        t = N_SYN - 1 - j\n        self.VF.insert(0, self.pool.acquire())\n        self.p2_sizes.append((len(self.VF), len(self.VG)))\n        self.pool.resize(self.VF, B[\'LVF\'][j])\n        self.pool.resize(self.VG, B[\'LVG\'][j])\n        c = self.stash.pop(t)\n        _to_tower(b, c)\n        for i in range(B[\'M2\'][j]):\n            self._mul(c, self.VG[i], self.VF[i], MUL_TK_K)\n        _from_tower(b, c)\n        inv = tower.TowerInverse(b, self.VG[0], out_basis=\'tower\')\n        y = inv.compute()\n        self._mul(self.VF[0], y, c, MUL_KT_K)\n        inv.uncompute()\n        inv.release()\n        self.pool.release(c)\n        for i in range(B[\'W2\'][j]):\n            b.cswap(self.swapbits[t], self.VF[i], self.VG[i])\n\n    def _p2_rev(self, j):\n        b, B = self.b, BOUNDS\n        t = N_SYN - 1 - j\n        for i in reversed(range(B[\'W2\'][j])):\n            b.cswap(self.swapbits[t], self.VF[i], self.VG[i])\n        c = self.pool.acquire()\n        inv = tower.TowerInverse(b, self.VG[0], out_basis=\'tower\')\n        y = inv.compute()\n        self._mul(self.VF[0], y, c, MUL_KT_K)\n        inv.uncompute()\n        inv.release()\n        _to_tower(b, c)\n        for i in reversed(range(B[\'M2\'][j])):\n            self._mul(c, self.VG[i], self.VF[i], MUL_TK_K)\n        _from_tower(b, c)\n        self.stash[t] = c\n        nvf, nvg = self.p2_sizes.pop()\n        self.pool.resize(self.VG, nvg)\n        self.pool.resize(self.VF, nvf)\n        self.pool.release(self.VF.pop(0))\n\n    # ------------------------------------------------------------------ Forney\n    def _forney(self):\n        b = self.b\n        gf = km.GF2Field(DEG)\n        alpha = gf.primitive_element\n        order = (1 << DEG) - 1\n        vg, vf = self.VG, self.VF\n        m = len(self.E)\n        mult = [(vg[i], gf.pow(alpha, i % order)) for i in range(1, len(vg))]\n        mult += [(vf[i], gf.pow(alpha, (i - 70) % order)) for i in range(len(vf))\n                 if (i - 70) % order]\n        pairs = ([(vg[i], vg[1]) for i in range(3, len(vg), 2)]\n                 + [(vg[1], vg[0])]\n                 + [(vg[i], vg[0]) for i in range(2, len(vg), 2)]\n                 + [(vf[i], vf[0]) for i in range(1, len(vf))])\n        src = km.array([q for s, _ in pairs for q in s])\n        dst = km.array([q for _, d in pairs for q in d])\n        rsrc = km.array([q for s, _ in reversed(pairs) for q in s])\n        rdst = km.array([q for _, d in reversed(pairs) for q in d])\n        is_error = b.alloc_bits(1)\n        forney_step = b.mark(\'forney_step\')\n        for j in range(m):\n            with forney_step:\n                b.bit_store(is_error[0], True, control=self.E[j])\n                b.push_condition(is_error[0])\n                b.cx(src, dst)\n                zt = ZeroTest(b, vg[0])\n                is_zero = zt.compute()\n                inv = tower.TowerInverse(b, vg[1], gate=is_zero)\n                y = inv.compute()\n                b.gf2_phase_by_product(vf[0], y, mask=self.E[j], field=gf)\n                inv.uncompute()\n                inv.release()\n                zt.uncompute()\n                b.cx(rsrc, rdst)\n                b.pop_condition()\n                b.bit_store(is_error, False)\n                if j + 1 < m:\n                    for reg, step in mult:\n                        b.gf2_imul(step, target=reg, field=gf)\n        if m > 1:\n            for reg, step in mult:\n                b.gf2_idiv(gf.pow(step, m - 1), target=reg, field=gf)\n        b.free(is_error)\n\n    # ------------------------------------------------------------------ driver\n    def build(self):\n        b = self.b\n        self.scratch = b.alloc_qubits(1)[0]\n        self.delta = b.alloc_qubits(DELTA_BITS)\n        b.x(self.delta[0])                                   # delta = 1\n        f0 = b.alloc_qubits(DEG)\n        b.x(f0[0])                                           # f = 1\n        self.F = [f0]\n        self.G = [self.S[N_SYN - 1 - i] for i in range(N_SYN)]\n        self.stash, self.swapbits, self.p1_sizes, self.p2_sizes = {}, {}, [], []\n        with b.mark(\'phase1\'):\n            for t in range(N_SYN):\n                self._p1(t)\n        one = self.pool.acquire()\n        b.x(one[0])\n        self.VF, self.VG = [], [one]\n        with b.mark(\'phase2\'):\n            for j in range(N_SYN):\n                self._p2(j)\n        with b.mark(\'forney\'):\n            self._forney()\n        with b.mark(\'phase2_rev\'):\n            for j in reversed(range(N_SYN)):\n                self._p2_rev(j)\n        assert not self.VF and len(self.VG) == 1\n        b.x(self.VG[0][0])\n        self.pool.release(self.VG.pop())\n        with b.mark(\'phase1_rev\'):\n            for t in reversed(range(N_SYN)):\n                self._p1_rev(t)\n        assert len(self.F) == 1 and self.F[0] is f0\n        assert all(a is b for a, b in zip(self.G, [self.S[N_SYN - 1 - i] for i in range(N_SYN)]))\n        b.x(f0[0])\n        b.free(f0)\n        b.x(self.delta[0])\n        b.free(self.delta)\n        for reg in self.pool.free:\n            b.free(reg)\n        b.free(self.scratch)\n\n\ndef build_circuit(m=4095, n=N_SYN, degree=DEG):\n    assert n == N_SYN and degree == DEG\n    builder = km.CircuitBuilder()\n    syndromes = [builder.create_quantum_register(degree, name=f\'S{i}\') for i in range(n)]\n    error_mbuc = [builder.create_classical_register(degree, name=f\'E{j}\') for j in range(m)]\n    TwoPhaseDecoder(builder, syndromes, error_mbuc).build()\n    return builder.finish_circuit()\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'output\', nargs=\'?\', default=\'dqi.kmb\')\n    p.add_argument(\'--m\', type=int, default=4095)\n    a = p.parse_args()\n    c = build_circuit(a.m)\n    c.to_file(a.output, format=\'kmb\')\n    print(f\'{a.output}: {c.operation_statistics().num_toffoli_like:,} Toffolis, {c.num_qubits} qubits\')\n\n\nif __name__ == \'__main__\':\n    main()\n')
(WORK / 'bounds_k20.json').write_text('{"AF": [70, 70, 70, 70, 69, 69, 68, 68, 67, 67, 66, 66, 65, 65, 64, 64, 63, 63, 62, 62, 61, 61, 60, 60, 59, 59, 58, 58, 57, 57, 56, 56, 55, 55, 54, 54, 53, 53, 52, 52, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51], "AG": [70, 70, 70, 70, 69, 69, 68, 68, 67, 67, 66, 66, 65, 65, 64, 64, 63, 63, 62, 62, 61, 61, 60, 60, 59, 59, 58, 58, 57, 57, 56, 56, 55, 55, 54, 54, 53, 53, 52, 52, 51, 51, 50, 50, 50, 49, 48, 48, 47, 47, 46, 46, 45, 45, 44, 44, 43, 43, 42, 42, 41, 41, 40, 40, 39, 39, 38, 37, 37, 36], "W": [70, 69, 70, 70, 69, 69, 68, 68, 67, 67, 66, 66, 65, 65, 64, 64, 63, 63, 62, 62, 61, 61, 60, 60, 59, 59, 58, 58, 57, 57, 56, 56, 55, 55, 54, 54, 53, 53, 52, 52, 51, 51, 50, 50, 50, 49, 48, 48, 47, 47, 46, 46, 45, 45, 44, 44, 43, 43, 42, 42, 41, 41, 40, 40, 39, 39, 38, 0, 37, 0], "R": [70, 70, 69, 69, 68, 68, 67, 67, 66, 66, 65, 65, 64, 64, 63, 63, 62, 62, 61, 61, 60, 60, 59, 59, 58, 58, 57, 57, 56, 56, 55, 55, 54, 54, 53, 53, 52, 52, 51, 51, 50, 50, 49, 49, 48, 48, 47, 47, 46, 46, 45, 45, 44, 44, 43, 43, 42, 42, 41, 41, 40, 40, 39, 39, 38, 38, 37, 37, 36, 36], "VF": [1, 2, 2, 3, 3, 4, 5, 5, 6, 6, 6, 7, 8, 8, 9, 9, 10, 10, 10, 11, 11, 12, 13, 13, 14, 14, 15, 15, 16, 16, 17, 17, 17, 18, 19, 19, 20, 20, 20, 21, 22, 22, 23, 23, 23, 24, 25, 25, 26, 26, 26, 27, 27, 28, 28, 29, 30, 30, 31, 31, 32, 32, 33, 33, 33, 34, 34, 35, 36, 36], "VG": [1, 2, 2, 3, 3, 4, 5, 5, 6, 6, 6, 7, 8, 8, 9, 9, 10, 10, 10, 11, 11, 12, 13, 13, 14, 14, 15, 15, 16, 16, 17, 17, 17, 18, 19, 19, 20, 20, 20, 21, 22, 22, 23, 23, 23, 24, 25, 25, 26, 26, 26, 27, 27, 28, 28, 29, 30, 30, 31, 31, 32, 32, 33, 33, 33, 34, 34, 35, 36, 36], "M2": [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9, 10, 10, 11, 11, 12, 12, 13, 13, 14, 14, 15, 15, 16, 16, 17, 17, 18, 18, 19, 19, 20, 20, 21, 21, 22, 22, 23, 23, 24, 24, 25, 25, 26, 26, 27, 27, 28, 28, 29, 29, 30, 30, 31, 31, 32, 32, 33, 33, 34, 34, 35, 35], "W2": [0, 2, 0, 3, 3, 4, 5, 5, 6, 6, 6, 7, 8, 8, 9, 9, 10, 10, 10, 11, 11, 12, 13, 13, 14, 14, 15, 15, 16, 16, 17, 17, 17, 18, 19, 19, 20, 20, 20, 21, 22, 22, 23, 23, 23, 24, 25, 25, 26, 26, 26, 27, 27, 28, 27, 29, 30, 30, 31, 31, 32, 32, 33, 33, 33, 34, 34, 35, 36, 36], "LVF": [1, 2, 2, 3, 3, 4, 5, 5, 6, 6, 6, 7, 8, 8, 9, 9, 10, 10, 10, 11, 11, 12, 13, 13, 14, 14, 15, 15, 16, 16, 17, 17, 17, 18, 19, 19, 20, 20, 20, 21, 22, 22, 23, 23, 23, 24, 25, 25, 26, 26, 26, 27, 27, 28, 28, 29, 30, 30, 31, 31, 32, 32, 33, 33, 33, 34, 34, 35, 36, 36], "LVG": [1, 2, 2, 3, 3, 4, 5, 5, 6, 6, 6, 7, 8, 8, 9, 9, 10, 10, 10, 11, 11, 12, 13, 13, 14, 14, 15, 15, 16, 16, 17, 17, 17, 18, 19, 19, 20, 20, 20, 21, 22, 22, 23, 23, 23, 24, 25, 25, 26, 26, 26, 27, 27, 28, 28, 29, 30, 30, 31, 31, 32, 32, 33, 33, 33, 34, 34, 35, 36, 36], "delta_range": [-2, 31]}')
print('sources written')

In [ ]:
# 3. Build the circuit (about 1-2 minutes)
import hashlib
env = {**os.environ, 'PYTHONPATH': os.pathsep.join([str(comp / 'kickmix'), str(comp / 'verifiers'), str(WORK),
                                                     os.environ.get('PYTHONPATH', '')])}
circuit = WORK / 'dqi.kmb'
out = subprocess.run([sys.executable, str(WORK / 'dqi_decoder.py'), str(circuit)], capture_output=True, text=True,
                     env=env, cwd=WORK)
print(out.stdout, out.stderr)
sha = hashlib.sha256(circuit.read_bytes()).hexdigest()
print('sha256', sha, 'matches local build:', sha == 'f296d406aaab6fc34e6ec17c480baaf25781d42cd65900ad351fd4a275f02af9')

In [ ]:
# 4. Official verifier, all 102,400 shots (a few minutes)
verifier = comp / 'verifiers' / 'dqi' / 'verify_submission_dqi.py'
out = subprocess.run([sys.executable, str(verifier), str(circuit)], capture_output=True, text=True, env=env)
if 'MemoryError' in out.stderr + out.stdout:
    # reaction_depth() (reported, not scored) needs a lot of RAM; rerun with it switched off.
    patched = WORK / 'verify_nodepth.py'
    patched.write_text(verifier.read_text().replace("circuit.reaction_depth()", "None"))
    out = subprocess.run([sys.executable, str(patched), str(circuit)], capture_output=True, text=True,
                         env={**env, 'PYTHONPATH': env['PYTHONPATH'] + os.pathsep + str(verifier.parent)})
    patched.unlink()
try:
    summary = json.loads(out.stdout)
except json.JSONDecodeError:
    print(out.stdout[-3000:], out.stderr[-3000:])
    raise
print({k: summary.get(k) for k in ('status', 'score', 'toffoli_count', 'num_qubits', 'value_errors', 'phase_errors',
                                   'repetition_factor')})
assert summary['status'] == 'PASS', out.stdout[-3000:] + out.stderr[-3000:]
print('ratio vs reference score 2.4439e13: x%.3f' % (2.4439463583672930e13 / summary['score']))

In [ ]:
# 5. Write submission.zip: everything from the source zip, with dqi.kmb replaced
target = WORK / 'submission.zip'
with zipfile.ZipFile(target, 'w', zipfile.ZIP_DEFLATED) as dst:
    if SOURCE.is_dir():                                   # Kaggle may show an attached zip as a folder
        names = [str(f.relative_to(SOURCE)) for f in SOURCE.rglob('*') if f.is_file()]
        assert 'dqi.kmb' in names, names
        for name in names:
            if name != 'dqi.kmb':
                dst.write(SOURCE / name, name)
    else:
        with zipfile.ZipFile(SOURCE) as src:
            names = src.namelist()
            assert 'dqi.kmb' in names, names
            for name in names:
                if name != 'dqi.kmb':
                    dst.writestr(src.getinfo(name), src.read(name))
    dst.write(circuit, 'dqi.kmb')
circuit.unlink()                                          # keep only submission.zip in the output
with zipfile.ZipFile(target) as z:
    for info in z.infolist():
        print(f'{info.file_size:>12,}  {info.filename}')
print('wrote', target, round(target.stat().st_size / 1e6, 1), 'MB')